# 02 · Data Quality

Raw TLC data has known dirt: zero/negative fares, zero-distance trips, dropoff-before-pickup,
implausible passenger counts, and **cross-month leakage** (rows whose pickup year isn't 2023).

We apply the required filters and **quantify each rule** so the cleaning is auditable, not a black box.

### Required filters
```
fare_amount   > 0
trip_distance > 0
tpep_dropoff_datetime > tpep_pickup_datetime
passenger_count BETWEEN 1 AND 6
EXTRACT(YEAR FROM tpep_pickup_datetime) = 2023   -- drops leakage
```

In [ ]:
import duckdb, pandas as pd
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

BASE = 'https://d37ci6vzurychx.cloudfront.net/trip-data'
SRC  = f'{BASE}/yellow_tripdata_2023-01.parquet'   # one month for speed
# Full year:  SRC = f'{BASE}/yellow_tripdata_2023-*.parquet'
print('Source:', SRC)

## 1. How many rows fail each rule?
We measure rules **independently** against the raw data so we can see each one's impact
(rules overlap, so these won't sum to the total dropped).

In [ ]:
checks = {
  'fare_amount <= 0'              : 'fare_amount <= 0',
  'trip_distance <= 0'           : 'trip_distance <= 0',
  'dropoff <= pickup'            : 'tpep_dropoff_datetime <= tpep_pickup_datetime',
  'passenger_count NOT 1..6'     : '(passenger_count IS NULL OR passenger_count < 1 OR passenger_count > 6)',
  'pickup year <> 2023'          : 'EXTRACT(YEAR FROM tpep_pickup_datetime) <> 2023',
}

total = con.sql(f"SELECT COUNT(*) FROM read_parquet('{SRC}')").fetchone()[0]
rows = []
for label, pred in checks.items():
    bad = con.sql(f"SELECT COUNT(*) FROM read_parquet('{SRC}') WHERE {pred}").fetchone()[0]
    rows.append({'rule': label, 'rows_failing': bad, 'pct': round(100*bad/total, 3)})

dq = pd.DataFrame(rows).sort_values('rows_failing', ascending=False)
print(f'Raw rows: {total:,}')
dq

## 2. Combined effect — how many survive all filters?

In [ ]:
clean = con.sql(f"""
  SELECT COUNT(*) FROM read_parquet('{SRC}')
  WHERE fare_amount > 0
    AND trip_distance > 0
    AND tpep_dropoff_datetime > tpep_pickup_datetime
    AND passenger_count BETWEEN 1 AND 6
    AND EXTRACT(YEAR FROM tpep_pickup_datetime) = 2023
""").fetchone()[0]

print(f'Raw     : {total:,}')
print(f'Clean   : {clean:,}')
print(f'Dropped : {total-clean:,}  ({100*(total-clean)/total:.2f}%)')

## 3. A reusable clean view
We register the filtered data as a **view** so notebooks `03` and `04` build on clean data
without copying it. Swap `SRC` to the full-year glob and everything downstream still works.

In [ ]:
con.sql(f"""
  CREATE OR REPLACE VIEW trips_clean AS
  SELECT * FROM read_parquet('{SRC}')
  WHERE fare_amount > 0
    AND trip_distance > 0
    AND tpep_dropoff_datetime > tpep_pickup_datetime
    AND passenger_count BETWEEN 1 AND 6
    AND EXTRACT(YEAR FROM tpep_pickup_datetime) = 2023
""")
con.sql("SELECT COUNT(*) AS clean_rows FROM trips_clean").df()

## Takeaways
- Each rule is **quantified**, so cleaning decisions are defensible.
- The biggest drops are usually zero-distance and out-of-range passenger counts.
- Cross-month leakage is small but real — the year filter removes December-2022 / January-2024 bleed.
- Next: **`03_schema_build`** — turn `trips_clean` into a star schema.